# Accessing the Digital Earth Normalised Radar Backscatter Product for Sentinel-1 (Collection 0) Using the Digital Earth Australia Sandbox

This notebook shows a number of additional features that can be taken advantage of when loading through the Digital Earth Australia Sandbox, which is directly connected to Digital Earth Australia's Open Data Cube.

## Important Datacube terminology
Data stored in an Open Data Cube has three important components:

* **Product**: A structure for organising all datasets in a single group.
* **Dataset**: A single spatio-temporal dataset, which would contain multiple measurements.
* **Measurement**: A single data measurement associated with a dataset, such as a single band.

For the Normalised Radar Backscatter data, it is important to note that a single *dataset* corresponds to a single *burst* from a larger acquisition.

## Set-up

Note that to use this notebook, you must be in the Digital Earth Australia production Sandbox, and you must have completed the [required steps](https://docs.dev.dea.ga.gov.au/services/dea-sandbox/dev_database_from_prod_jupyterhub/) to connect to the dev Datacube database.

Instructions are only available to Geoscience Australia employees.

### Import required libraries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from odc.geo import BoundingBox
import odc.geo.xr
import pathlib
import xarray as xr
import datacube

### Environment set up

In [ ]:
dc = datacube.Datacube(env="dev", app="S1_Backscatter_Demo")

## Interrogating products and datasets

### View available products

In [ ]:
dc_products = dc.list_products()
dc_products.loc[dc_products["name"].str.contains("ga_s1_nrb_iw")]

### View product measurements

In [ ]:
dc_measurements = dc.list_measurements()
dc_measurements.loc['ga_s1_nrb_iw_vv_vh_0']

### View product metadata

Products contain many additional metadata fields that can be valuable for filtering.

In [ ]:
dc.index.products.get_by_name("ga_s1_nrb_iw_vv_vh_0").fields

### View dataset metadata

The actual metadata values are available on a dataset level. For this example, we show the metadata associated with a specific item, [ga_s1a_nrb_0-1-0_T002-003369-IW3_20250416T203421Z](https://explorer.dev.dea.ga.gov.au/products/ga_s1_nrb_iw_vv_vh_0/datasets/222a14f0-6fef-5455-8198-1e5e6e2a2d41), which has a dataset id of `222a14f0-6fef-5455-8198-1e5e6e2a2d41`

In [ ]:
single_dataset = dc.index.datasets.get("222a14f0-6fef-5455-8198-1e5e6e2a2d41")

single_dataset.metadata.search_fields

## Basic search and load

When working with the Open Data Cube datacube API, it is possible to do searching and loading with a single call of the `dc.load()` function.

### Set search and load parameters

In [ ]:
# Products to query
product_to_load = "ga_s1_nrb_iw_vv_vh_0"

# Area of interest
aoi_bbox = BoundingBox(
    left=147.0,
    bottom=-42.5,
    right=148.0,
    top=-41.5,
    crs="EPSG:4326"
)

# Date range of interest
start_date = "2024-06-01"
end_date = "2025-06-30"

# Measurements to load
measurements_to_load = ["VV_gamma0", "VH_gamma0", "mask"]

# CRS and resolution
output_crs = "EPSG:3577"
output_res = 400 # 20m is the native resolution of the data. Here, we resample to 400m to save memory

# Property or function to group by. "solar_day" is already built into odc-stac
groupy_by_operation = "solar_day"

### Run load

Note that the load function takes:

* `product`: the product, or list of products to load from.
* `measurements`: the measurements to load from the data (e.g. VV). If not specified, all measurements will be loaded.
* `lat`: a tuple of latitude values (bottom, top)
* `lon`: a tuple of longitude values (left, right)
* `output_crs`: the coordinate reference system (CRS) to project loaded data to. If not specified, the data's native CRS will be used.
* `resolution`: the resolution to load the data at, in the same units as the chosen CRS. If not specified, the data's native resolution will be used.
* `time`: a tuple of (start, end) indicating the date range.
* `group_by`: how to group loaded data. Solar day is a sensible default.

See the [API](https://opendatacube.readthedocs.io/en/latest/api/core-classes/datacube.html#datacube.Datacube.load) for more information

In [ ]:
ds = dc.load(
    product=product_to_load,
    measurements=measurements_to_load,
    lat=(aoi_bbox.bottom, aoi_bbox.top),
    lon=(aoi_bbox.left, aoi_bbox.right),
    output_crs=output_crs,
    resolution=(output_res, -output_res),
    time=(start_date, end_date),
    group_by=groupy_by_operation,
    dask_chunks={}
)

### View the lazy-loaded xarray

In [ ]:
ds

### Compute the lazy loaded xarray
To save time and memory for this demonstration, we'll only compute the first three time-steps.

In [ ]:
ds = ds.isel(time=range(0,3)).compute()

ds

### Visualise

This step shows how to visualise the first three time-steps from the xarray.
After grouping by solar day, you can see how the Sentinel-1 orbit tracks influence where data is and isn't captured on a given day.

In [ ]:
ds.isel(time=range(0,3))["VV_gamma0"].plot.imshow(col="time", col_wrap=3, robust=True, cmap="Greys_r")

## Advanced approaches

### Searching before loading

While `dc.load()` can combine searching and loading, the `datacube` library also has a `find_datasets()` fucntion, which returns a list of datasets that can then be loaded.
This can be helpful when wanting to filter datasets on specific metadata properties prior to loading.

See the [API](https://opendatacube.readthedocs.io/en/latest/api/core-classes/datacube.html#datacube.Datacube.find_datasets) for more information.

In [ ]:
datasets = dc.find_datasets(
    product=product_to_load,
    lat=(aoi_bbox.bottom, aoi_bbox.top),
    lon=(aoi_bbox.left, aoi_bbox.right),
    time=(start_date, end_date),
)

print(f"Found {len(datasets)} datasets")

### Viewing dataset-level metadata

Before loading, it can be valuable to understand the general metadata properties of the returned datasets.
For example, are your datasets primarily captured in descending or ascending passes?

In [ ]:
ascending_datasets = [dataset for dataset in datasets if dataset.metadata.orbit_state=="ascending"]
print(f"Number of datasets with ascending orbit state: {len(ascending_datasets)}")

descending_datasets = [dataset for dataset in datasets if dataset.metadata.orbit_state=="descending"]
print(f"Number of datasets with descending orbit state: {len(descending_datasets)}")

Alternatively, you could look at the number of unique absolute orbits (tracks) or scenes across your items.

In [ ]:
unique_relative_orbits = set([dataset.metadata.relative_orbit for dataset in datasets])
print(f"The identified items come from {len(unique_relative_orbits)} unique relative orbits")
print(f"The unique relative orbit values are: {unique_relative_orbits}\n")

unique_absolute_orbits = set([dataset.metadata.absolute_orbit for dataset in datasets])
print(f"The identified items come from {len(unique_absolute_orbits)} unique absolute orbits")
print(f"The unique absolute orbit values are: {unique_absolute_orbits}\n")

unique_scene_ids = set([dataset.metadata.scene_id for dataset in datasets])
print(f"The identified items come from {len(unique_scene_ids)} unique scenes")
print(f"The unique scene ids are: {unique_scene_ids}")


### Loading found datasets

Having run the `dc.find_datasets()` function, it is possible to simplify the `dc.load()` function by supplying the list of datasets to load, as the product and time-range are no longer required.

In [ ]:
ds_from_find = dc.load(
    datasets=datasets,
    measurements=measurements_to_load,
    lat=(aoi_bbox.bottom, aoi_bbox.top),
    lon=(aoi_bbox.left, aoi_bbox.right),
    output_crs=output_crs,
    resolution=(output_res, -output_res),
    group_by=groupy_by_operation,
    dask_chunks={}
)

ds_from_find

### Loading datasets for known metadata values

If you already know the value of the metadata you wish to load on, it is possible to provide this to `dc.load()` directly by specifying the metadata name as a key-word argument.
See the [01_product_info](01_product_info.ipynb) notebook for a table containing key metadata that can be used when searching and loading.

Below, we show how to filter items for a single known `scene_id`:

In [ ]:
ds_scene = dc.load(
    product=product_to_load,
    measurements=measurements_to_load,
    lat=(aoi_bbox.bottom, aoi_bbox.top),
    lon=(aoi_bbox.left, aoi_bbox.right),
    output_crs=output_crs,
    resolution=(output_res, -output_res),
    time=(start_date, end_date),
    group_by=groupy_by_operation,
    scene_id="S1A_IW_SLC__1SDV_20241115T191815_20241115T191842_056569_06EFD5_BBCA",
    dask_chunks={}
)

ds_scene

### Grouping by metadata other than solar day

It is possible to provide a custom `groupby` function which can be passed to `dc.load()`.
The below example groups items by scene, rather than by solar day.

In [ ]:
from datacube.api.query import GroupBy

def group_by_scene_id():
    def by_scene_id(ds):
        return ds.metadata.scene_id

    def coord_for_group(datasets_group):
        t = min(ds.center_time for ds in datasets_group)
        return t

    return GroupBy(
        group_by_func=by_scene_id,
        dimension="time",
        units="seconds since 1970-01-01 00:00:00",
        sort_key=lambda ds: ds.center_time,
        group_key=coord_for_group,
    )

ds_by_scene = dc.load(
    product=product_to_load,
    measurements=measurements_to_load,
    lat=(aoi_bbox.bottom, aoi_bbox.top),
    lon=(aoi_bbox.left, aoi_bbox.right),
    output_crs=output_crs,
    resolution=(output_res, -output_res),
    time=(start_date, end_date),
    group_by=group_by_scene_id(),
    dask_chunks={}
)

The effect of grouping by scene is that there are a different number of time-steps, one per scene.

In [ ]:
ds_by_scene

The impact of grouping by scene is also visible when displaying the data.
There are two scenes for our area of interest that were captured on the same day. 
These are now displayed in separate time-steps.

In [ ]:
ds_by_scene.isel(time=range(0,3))["VV_gamma0"].plot.imshow(col="time", col_wrap=3, robust=True, cmap="Greys_r")